# Instantaneous velocity and speed from cell tracks

Computes **signed instantaneous velocities** and per-cell **speed statistics**
from time-resolved single-cell tracks along the apicobasal axis.

- The velocity between two consecutive timepoints is the change in position
  divided by the elapsed time, assigned to the midpoint between them.
- Velocities are kept **signed**. Positions are distances from the apical
  surface (negative = basal), so negative velocities mean basal movement.
- Units: positions in µm, time in min, so velocities and speeds are in µm/min.

**Input:** a CSV with a `Time` column (min) and one column per cell (position in µm).
Missing timepoints can be left empty.

Edit the **Config** cell and run all cells. Outputs are written to `OUT_DIR`.

Used in Cruz et al. (2026), *Development*.


## Imports

In [ ]:
import os

import numpy as np
import pandas as pd

## Config — edit here

In [ ]:
FILE_PATH = "example_data/example_tracks.csv"   # Time column + one column per cell
SEP       = ";"          # column separator of the input CSV
DECIMAL   = ","          # decimal mark of the input CSV
OUT_DIR   = "results/instantaneous_velocity"   # where output CSVs are saved


## Functions

In [ ]:
def compute_velocities(df):
    """Signed instantaneous velocities at the midpoint between consecutive timepoints."""
    time = df.index.to_numpy(dtype=float)
    pos = df.to_numpy(dtype=float)
    dt = np.diff(time)[:, None]
    vel = np.diff(pos, axis=0) / dt
    midpoints = (time[:-1] + time[1:]) / 2
    out = pd.DataFrame(vel, index=midpoints, columns=df.columns)
    out.index.name = "Time_midpoint_min"
    return out


def velocity_summary(velocity_df):
    """Per-cell signed-velocity statistics (NaN ignored)."""
    return pd.DataFrame({
        "mean_velocity": velocity_df.mean(),
        "std_velocity":  velocity_df.std(),
        "min_velocity":  velocity_df.min(),
        "max_velocity":  velocity_df.max(),
    })


def speed_summary(df):
    """Per-cell speed metrics: mean speed, net displacement, total path length, average speed.

    Distances use the actual time steps, so no fixed frame interval is assumed.
    mean_speed: mean of |step| / dt over steps; avg_speed: total path / track duration."""
    time = df.index.to_numpy(dtype=float)
    dt = np.diff(time)
    metrics = {}
    for cell in df.columns:
        pos = df[cell].to_numpy(dtype=float)
        finite = np.isfinite(pos)
        step = np.diff(pos)
        net = pos[finite][-1] - pos[finite][0] if finite.sum() >= 2 else np.nan
        total_dist = np.nansum(np.abs(step))
        # duration of this cell's own track (first to last tracked timepoint)
        total_time = time[finite][-1] - time[finite][0] if finite.sum() >= 2 else np.nan
        metrics[cell] = {
            "mean_speed_um_min":   np.nanmean(np.abs(step) / dt),
            "net_displacement_um": net,
            "total_distance_um":   total_dist,
            "avg_speed_um_min":    total_dist / total_time if total_time > 0 else np.nan,
        }
    return pd.DataFrame(metrics).T

## Load data

In [ ]:
df = pd.read_csv(FILE_PATH, sep=SEP, decimal=DECIMAL).set_index("Time")
print(f"Loaded {df.shape[1]} cells x {df.shape[0]} timepoints")

## Signed instantaneous velocities

In [ ]:
velocity_df = compute_velocities(df)
velocity_df.round(4)

## Velocity summary (per cell)

In [ ]:
vel_summary = velocity_summary(velocity_df)
vel_summary.round(4)

## Speed summary (per cell)

In [ ]:
spd_summary = speed_summary(df)
spd_summary.round(4)

## Save outputs

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
velocity_df.to_csv(os.path.join(OUT_DIR, "instantaneous_velocities.csv"))
vel_summary.to_csv(os.path.join(OUT_DIR, "velocity_summary.csv"))
spd_summary.to_csv(os.path.join(OUT_DIR, "speed_summary.csv"))
print(f"Saved instantaneous_velocities.csv, velocity_summary.csv and speed_summary.csv to: {OUT_DIR}")
